# Colab 2 · Orchestration Patterns in Depth
### Day 19 — Agent Orchestration with AutoGen Studio & Semantic Kernel

Colab 1 used the simplest pattern (RoundRobin). Now you'll wire the **same research team four different ways** and watch how the *control flow* changes — then peek at the **same idea in Semantic Kernel**.

**You will build:**
1. **SelectorGroupChat** — an LLM decides who speaks next.
2. **Swarm** — agents hand off to each other directly.
3. **GraphFlow** — a deterministic researcher → writer → reviewer graph.
4. A **function tool** the researcher calls to delegate real work.
5. A **Semantic Kernel** sequential-orchestration mini-example.

⏱️ ~60 min including the extension tasks at the end.

> The patterns are the lesson. AutoGen, Semantic Kernel and the Microsoft Agent Framework all expose this same family — RoundRobin/Sequential, Selector/GroupChat, Swarm/Handoff, Graph, Magentic.

## 0 · Setup

In [30]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"
print("AutoGen installed.")

AutoGen installed.


In [31]:
import os
from getpass import getpass
from google.colab import userdata
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # getpass("Paste your OpenAI API key: ")

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Ready.")

Ready.


### Three specialists we'll reuse

Notice the **descriptions** — Selector and Swarm route work based on them, so they have to be specific and non-overlapping.

In [32]:
def make_specialists():
    planner = AssistantAgent(
        name="planner",
        model_client=model_client,
        description="Breaks a topic into 2-3 concrete sub-questions to research.",
        system_message="You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short.",
    )
    researcher = AssistantAgent(
        name="researcher",
        model_client=model_client,
        description="Answers factual sub-questions with concise bullet points.",
        system_message="You answer the planner's sub-questions with short factual bullets.",
    )
    writer = AssistantAgent(
        name="writer",
        model_client=model_client,
        description="Turns research bullets into a tight 4-sentence summary, ending with APPROVE.",
        system_message="Write a tight 4-sentence summary from the research. End your message with APPROVE.",
    )
    return planner, researcher, writer

print("Specialist factory ready.")

Specialist factory ready.


## 1 · SelectorGroupChat — let an LLM route

Instead of a fixed order, a **SelectorGroupChat** uses a model to pick *who should act next* based on the conversation and each agent's `description`. Good when the next best speaker depends on what just happened.

Key knobs: it needs its own `model_client` to do the routing, and `allow_repeated_speaker=False` stops one agent from monopolising the floor.

In [33]:
from autogen_agentchat.teams import SelectorGroupChat

planner, researcher, writer = make_specialists()
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

selector_team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,        # the "router" brain
    termination_condition=termination,
    allow_repeated_speaker=False,
)

await Console(selector_team.run_stream(
    task="Topic: why are reusable cups better than disposable ones?"
))

---------- TextMessage (user) ----------
Topic: why are reusable cups better than disposable ones?
---------- TextMessage (planner) ----------
1. What are the environmental impacts of producing and disposing of reusable cups compared to disposable ones?  
2. How do the long-term cost savings of using reusable cups compare to continuously purchasing disposable cups?  
3. What are the health and safety benefits associated with using reusable cups versus disposables?  
---------- TextMessage (researcher) ----------
1. **Environmental Impacts:**
   - Reusable cups reduce waste: One reusable cup can replace hundreds of disposable cups over time.
   - Production of disposable cups requires significant resources, contributing to deforestation and increased carbon emissions.
   - Disposable cups often end up in landfills or oceans, leading to pollution and harm to wildlife.
   - Reusable cups can be made from sustainable materials, reducing overall environmental footprint.

2. **Long-Term Cost

TaskResult(messages=[TextMessage(id='c64ecb93-2528-433f-b4c3-82c53552adb4', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 33, 269432, tzinfo=datetime.timezone.utc), content='Topic: why are reusable cups better than disposable ones?', type='TextMessage'), TextMessage(id='efd9ac0d-31b6-4231-a085-46942cb54f78', source='planner', models_usage=RequestUsage(prompt_tokens=45, completion_tokens=60), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 35, 957047, tzinfo=datetime.timezone.utc), content='1. What are the environmental impacts of producing and disposing of reusable cups compared to disposable ones?  \n2. How do the long-term cost savings of using reusable cups compare to continuously purchasing disposable cups?  \n3. What are the health and safety benefits associated with using reusable cups versus disposables?  ', type='TextMessage'), TextMessage(id='686bea76-8d99-4922-a195-b9b6b6f39d4b', source='researcher', models_usa

Look at the speaker order in the transcript — it was **chosen at runtime**, not fixed. That's the difference from RoundRobin.

## 2 · Swarm — agents hand off to each other

In a **Swarm**, control is decentralised: each agent declares who it can **hand off** to via `handoffs=[...]`, and passes control with a handoff message. There's no central router — the agents themselves decide.

We'll build a tiny triage flow: a `triage` agent routes to either `billing` or `tech`, and those can hand back to the user when done.

In [34]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination

triage = AssistantAgent(
    name="triage",
    model_client=model_client,
    handoffs=["billing", "tech"],
    description="Front desk: routes the user to the right specialist.",
    system_message="Decide if the request is about billing or tech, then hand off to that agent.",
)
billing = AssistantAgent(
    name="billing",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles billing and refund questions.",
    system_message="Answer the billing question. If it's not billing, hand back to triage.",
)
tech = AssistantAgent(
    name="tech",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles technical troubleshooting.",
    system_message="Answer the tech question concisely, then say DONE.",
)

swarm = Swarm(
    participants=[triage, billing, tech],          # Swarm starts with the first agent
    termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(8),
)

await Console(swarm.run_stream(task="My app keeps crashing when I open the camera."))

---------- TextMessage (user) ----------
My app keeps crashing when I open the camera.
---------- ToolCallRequestEvent (triage) ----------
[FunctionCall(id='call_AN9haKg3YhAtcIXpqWfStxyb', arguments='{}', name='transfer_to_tech')]
---------- ToolCallExecutionEvent (triage) ----------
[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='transfer_to_tech', call_id='call_AN9haKg3YhAtcIXpqWfStxyb', is_error=False)]
---------- HandoffMessage (triage) ----------
Transferred to tech, adopting the role of tech immediately.
---------- ToolCallRequestEvent (tech) ----------
[FunctionCall(id='call_5pvVXPEqfde5H2bGymBSpLtz', arguments='{}', name='transfer_to_triage')]
---------- ToolCallExecutionEvent (tech) ----------
[FunctionExecutionResult(content='Transferred to triage, adopting the role of triage immediately.', name='transfer_to_triage', call_id='call_5pvVXPEqfde5H2bGymBSpLtz', is_error=False)]
---------- HandoffMessage (tech) ----------
Trans

TaskResult(messages=[TextMessage(id='a84cfa04-f2c2-4193-a9e0-17fdffd0cd65', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 42, 831495, tzinfo=datetime.timezone.utc), content='My app keeps crashing when I open the camera.', type='TextMessage'), ToolCallRequestEvent(id='4db411ed-8fba-464a-8f9b-fcd3cec6a5ca', source='triage', models_usage=RequestUsage(prompt_tokens=82, completion_tokens=12), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 43, 492485, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_AN9haKg3YhAtcIXpqWfStxyb', arguments='{}', name='transfer_to_tech')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='c70e1cfb-eb78-4094-86fe-3f47de5a23f3', source='triage', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 43, 495020, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='tra

Watch for the **HandoffMessage** in the transcript — that's one agent explicitly delegating to another. `HandoffTermination(target="user")` is another common stop condition when an agent hands control back to a human.

## 3 · GraphFlow — a deterministic workflow

When you need the **same path every time** (auditable, reproducible), use **GraphFlow**. You declare nodes and directed edges with `DiGraphBuilder`; execution follows the graph exactly.

Here: `planner → researcher → writer`, a fixed pipeline with no LLM routing brain.

In [35]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

planner, researcher, writer = make_specialists()

builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer)
builder.add_edge(planner, researcher).add_edge(researcher, writer)
graph = builder.build()

flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

await Console(flow.run_stream(task="Topic: the benefits of cycling to work."))

---------- TextMessage (user) ----------
Topic: the benefits of cycling to work.
---------- TextMessage (planner) ----------
1. What are the physical health benefits associated with commuting by bicycle compared to other modes of transportation?
2. How does cycling to work impact mental well-being and stress levels in employees?
3. What economic advantages do companies experience when encouraging cycling as a mode of transportation for their employees?
---------- TextMessage (researcher) ----------
1. **Physical Health Benefits of Cycling to Work:**
   - Improves cardiovascular fitness and endurance.
   - Aids in weight management and fat loss.
   - Strengthens muscles and enhances joint mobility.
   - Reduces the risk of chronic diseases (e.g., heart disease, diabetes).
   - Increases flexibility and balance.
   - Boosts immune system function.

2. **Impact of Cycling on Mental Well-Being:**
   - Reduces stress levels through physical activity and fresh air.
   - Enhances mood by prom

TaskResult(messages=[TextMessage(id='da45288b-22e0-486f-8862-6e612996e540', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 49, 129471, tzinfo=datetime.timezone.utc), content='Topic: the benefits of cycling to work.', type='TextMessage'), TextMessage(id='744f30eb-4772-4a67-8b6d-a433b887533b', source='planner', models_usage=RequestUsage(prompt_tokens=43, completion_tokens=57), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 50, 442451, tzinfo=datetime.timezone.utc), content='1. What are the physical health benefits associated with commuting by bicycle compared to other modes of transportation?\n2. How does cycling to work impact mental well-being and stress levels in employees?\n3. What economic advantages do companies experience when encouraging cycling as a mode of transportation for their employees?', type='TextMessage'), TextMessage(id='c5e1d61a-3ff5-4b44-a0f3-77ecbd3e406a', source='researcher', models_usage=RequestUsag

GraphFlow gives you **determinism**: the order is guaranteed by the graph, not decided by a model. That's exactly what you want for a compliance-sensitive or repeatable pipeline.

## 4 · A function tool the researcher can call

Delegation isn't only agent-to-agent — an agent can delegate to **code** via a tool. Define a plain Python function, pass it in `tools=[...]`, and the agent will call it when useful. (Here it's a stub; swap in a real search API at home.)

In [36]:
def web_search(query: str) -> str:
    """Look up a query and return a short text snippet. (Stub for the workshop.)"""
    canned = {
        "reusable cup co2": "A reusable cup typically breaks even vs. disposables after ~20-100 uses.",
        "default": "No exact match; returning a generic note that reusable goods amortise their footprint with use.",
    }
    return canned.get(query.lower().strip(), canned["default"])

researcher_with_tool = AssistantAgent(
    name="researcher",
    model_client=model_client,
    tools=[web_search],
    description="Researches facts, calling web_search when it needs evidence.",
    system_message="Use the web_search tool to find a figure, then report it in one bullet. End with APPROVE.",
)

from autogen_agentchat.teams import RoundRobinGroupChat
tool_team = RoundRobinGroupChat(
    [researcher_with_tool],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(4),
)
await Console(tool_team.run_stream(task="Find a figure on reusable cup CO2 break-even and report it."))

---------- TextMessage (user) ----------
Find a figure on reusable cup CO2 break-even and report it.
---------- ToolCallRequestEvent (researcher) ----------
[FunctionCall(id='call_MqMuRa8UGNoL6Fa7rqSguR16', arguments='{"query":"reusable cup CO2 break-even"}', name='web_search')]
---------- ToolCallExecutionEvent (researcher) ----------
[FunctionExecutionResult(content='No exact match; returning a generic note that reusable goods amortise their footprint with use.', name='web_search', call_id='call_MqMuRa8UGNoL6Fa7rqSguR16', is_error=False)]
---------- ToolCallSummaryMessage (researcher) ----------
No exact match; returning a generic note that reusable goods amortise their footprint with use.
---------- ToolCallRequestEvent (researcher) ----------
[FunctionCall(id='call_Rb6UfCEdrf3Y8K1kOJk9a2U0', arguments='{"query":"reusable cup carbon footprint break-even analysis"}', name='web_search')]
---------- ToolCallExecutionEvent (researcher) ----------
[FunctionExecutionResult(content='No exa

TaskResult(messages=[TextMessage(id='cd3a2319-4bf1-4443-a553-89c451a7c882', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 55, 619296, tzinfo=datetime.timezone.utc), content='Find a figure on reusable cup CO2 break-even and report it.', type='TextMessage'), ToolCallRequestEvent(id='d9815076-bed5-4bc5-b5dc-54555d0ff264', source='researcher', models_usage=RequestUsage(prompt_tokens=97, completion_tokens=20), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 56, 682556, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_MqMuRa8UGNoL6Fa7rqSguR16', arguments='{"query":"reusable cup CO2 break-even"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='a927d31c-6dd8-4119-ac85-d09bb4eef6a7', source='researcher', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 57, 56, 684942, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='No exact match; re

The transcript shows a **ToolCall** and its result — the agent delegated part of its job to your function. In production you'd scope tools to least privilege and validate their arguments.

## 5 · The same idea in Semantic Kernel

Semantic Kernel expresses these patterns too — its **Sequential** orchestration is the SK analogue of RoundRobin/GraphFlow-in-a-line. The code below shows the *shape* of SK agent orchestration.

> ⚠️ SK's agent-orchestration API is newer and evolving (and SK is in maintenance mode heading into the Microsoft Agent Framework). If an import path has moved, check the official Semantic Kernel docs — the **concept** is what transfers, not the exact symbol names.

In [37]:
%pip install -q -U semantic-kernel
print("Semantic Kernel installed.")

Semantic Kernel installed.


In [38]:
# The shape of an SK sequential orchestration: two agents, output of one feeds the next.
# Wrapped in try/except because SK's orchestration symbols move between versions.
import asyncio

try:
    from semantic_kernel.agents import ChatCompletionAgent
    from semantic_kernel.agents.orchestration.sequential import SequentialOrchestration
    from semantic_kernel.agents.runtime import InProcessRuntime
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service = OpenAIChatCompletion(ai_model_id="gpt-4o-mini")

    sk_writer = ChatCompletionAgent(
        name="writer", service=service,
        instructions="Write one short paragraph on the given topic.",
    )
    sk_editor = ChatCompletionAgent(
        name="editor", service=service,
        instructions="Tighten the paragraph you receive into two crisp sentences.",
    )

    orchestration = SequentialOrchestration(members=[sk_writer, sk_editor])
    runtime = InProcessRuntime()
    runtime.start()

    result = await orchestration.invoke(
        task="The benefits of walking meetings.", runtime=runtime
    )
    print(await result.get())
    await runtime.stop_when_idle()

except Exception as e:
    print("SK orchestration symbols may have moved in your installed version.")
    print("Concept: members=[writer, editor] run in sequence, output -> input.")
    print("Check https://learn.microsoft.com/semantic-kernel for the current API.")
    print("Error was:", type(e).__name__, e)

Walking meetings boost productivity and creativity by enhancing blood flow and mood, leading to clearer thinking and innovative ideas. They promote open communication in a relaxed setting while combating a sedentary lifestyle, making them a dynamic alternative to traditional meetings.


Notice the **identical mental model**: a list of agents, run in order, each one's output feeding the next. RoundRobin (AutoGen) ≈ Sequential (SK) ≈ a linear GraphFlow. Learn it once.

---
## 🚀 Extension tasks

### Extension 1 — Write a custom selector function
`SelectorGroupChat` accepts a `selector_func` that overrides the LLM router with your own logic. Write a function that **forces** `planner` to go first, then lets the model choose. (Signature: it receives the message history and returns the next speaker's name, or `None` to defer to the model.)

### Extension 2 — Add a conditional GraphFlow edge
Extend the graph from §3 with a **reviewer** node and a **conditional edge**: if the writer's output contains the word `REVISE`, loop back to the writer; otherwise finish. Use `DiGraphBuilder`'s conditional-edge support and a stop condition so it can't loop forever.

### Extension 3 — Nest a team inside a graph node
A node in a GraphFlow can itself be a **team**. Replace the single `researcher` node with a 2-agent `RoundRobinGroupChat` (researcher + fact-checker) and wire that team in as one node. This is *composition*: patterns nest inside patterns.

Scaffolds below.

### Extension-1

In [39]:
def force_planner_first(messages):
    """
    Force planner to speak first.
    After that let AutoGen's LLM router decide.
    """

    # First interaction
    if len(messages) <= 1:
        return "planner"

    # Hand control back to SelectorGroupChat
    return None

In [40]:
planner, researcher, writer = make_specialists()

termination = (
    TextMentionTermination("APPROVE")
    | MaxMessageTermination(10)
)

In [41]:
from autogen_agentchat.teams import SelectorGroupChat

team = SelectorGroupChat(
    participants=[
        planner,
        researcher,
        writer
    ],
    model_client=model_client,
    selector_func=force_planner_first,
    termination_condition=termination,
)

In [42]:
from autogen_agentchat.teams import SelectorGroupChat

team = SelectorGroupChat(
    participants=[
        planner,
        researcher,
        writer
    ],
    model_client=model_client,
    selector_func=force_planner_first,
    termination_condition=termination,
)

In [43]:
result = await team.run(
    task="Research electric vehicles and create a summary."
)

print(result)

messages=[TextMessage(id='d6b0e716-2144-4054-a75d-780f074b595e', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 58, 7, 174307, tzinfo=datetime.timezone.utc), content='Research electric vehicles and create a summary.', type='TextMessage'), TextMessage(id='20211817-8739-44ea-b9ef-f907bd0b922b', source='planner', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=51), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 58, 8, 397463, tzinfo=datetime.timezone.utc), content='1. What are the key technological advancements in electric vehicle batteries over the past five years?\n2. How do the environmental impacts of electric vehicles compare to traditional gasoline vehicles?\n3. What are the challenges and barriers to widespread electric vehicle adoption in urban areas?', type='TextMessage'), TextMessage(id='e6127054-7b35-4b35-98e0-915e7833cb8d', source='researcher', models_usage=RequestUsage(prompt_tokens=90, completion_tokens=

### Extension-3

In [44]:
fact_checker = AssistantAgent(
    name="fact_checker",
    model_client=model_client,
    description="Verifies facts produced by researcher.",
    system_message="""
    Verify claims made by the researcher.
    Identify inaccuracies or unsupported statements.
    """
)

In [45]:
from autogen_agentchat.teams import RoundRobinGroupChat

research_team = RoundRobinGroupChat(
    participants=[
        researcher,
        fact_checker
    ],
    termination_condition=MaxMessageTermination(4)
)

In [48]:
fact_checker = AssistantAgent(
    name="fact_checker",
    model_client=model_client,
    system_message="Verify researcher findings."
)

builder = DiGraphBuilder()

builder.add_node(planner)
builder.add_node(researcher)
builder.add_node(fact_checker)
builder.add_node(writer)

builder.add_edge(planner, researcher)
builder.add_edge(researcher, fact_checker)
builder.add_edge(fact_checker, writer)

graph = builder.build()

flow = GraphFlow(
    participants=[
        planner,
        researcher,
        fact_checker,
        writer
    ],
    graph=graph,
    termination_condition=MaxMessageTermination(12)
)

In [49]:
await Console(
    flow.run_stream(
        task="Research renewable energy adoption in India."
    )
)

---------- TextMessage (user) ----------
Research renewable energy adoption in India.
---------- TextMessage (planner) ----------
1. What government policies and incentives have been implemented to promote renewable energy adoption in India?
2. How do the levels of renewable energy adoption vary between urban and rural areas in India?
3. What are the main challenges India faces in scaling up its renewable energy capacity?
---------- TextMessage (researcher) ----------
### Renewable Energy Adoption in India

#### 1. Government Policies and Incentives to Promote Renewable Energy
- **National Solar Mission**: Aimed at increasing solar energy capacity to 100 GW by 2022, now extended to 300 GW.
- **Renewable Purchase Obligation (RPO)**: Mandates that utilities buy a certain percentage of their energy from renewable sources.
- **Tax Incentives**: Includes accelerated depreciation benefits, customs duty exemptions, and income tax holidays for renewable projects.
- **Financial Schemes**: Budge

TaskResult(messages=[TextMessage(id='4210fd45-1349-4005-8ec1-d9c266b84668', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 59, 33, 98560, tzinfo=datetime.timezone.utc), content='Research renewable energy adoption in India.', type='TextMessage'), TextMessage(id='30b19624-1062-475d-b48b-3dc6d56b210c', source='planner', models_usage=RequestUsage(prompt_tokens=109, completion_tokens=54), metadata={}, created_at=datetime.datetime(2026, 6, 22, 12, 59, 34, 277838, tzinfo=datetime.timezone.utc), content='1. What government policies and incentives have been implemented to promote renewable energy adoption in India?\n2. How do the levels of renewable energy adoption vary between urban and rural areas in India?\n3. What are the main challenges India faces in scaling up its renewable energy capacity?', type='TextMessage'), TextMessage(id='6d48203f-e2ef-45fc-901e-a87f2215f16a', source='researcher', models_usage=RequestUsage(prompt_tokens=551, completion

## Recap

You orchestrated one research team **five ways** and saw exactly how control flow differs:

* **Selector** — LLM picks the next speaker (dynamic).
* **Swarm** — agents hand off to each other (decentralised).
* **GraphFlow** — a fixed, auditable graph (deterministic).
* **Tools** — an agent delegates work to a function.
* **Semantic Kernel** — the same Sequential idea in the enterprise SDK.

Pair this with the decision matrix from the slides, then tackle the **capstone**: build your own delegating team in AutoGen Studio.

In [ ]:
await model_client.close()
print("Client closed. On to the capstone!")